In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [27]:
spark = SparkSession.builder.appName("Translink Bus Data").getOrCreate()

In [28]:
suffix = "year=2025/month=1/day=10"

realtime_df = spark.read.parquet(f"../data/spark_partitioned/realtime/{suffix}")    
position_df = spark.read.parquet(f"../data/spark_partitioned/position/{suffix}")
weather_df = spark.read.parquet(f"../data/spark_partitioned/weather/{suffix}")

stops = spark.read.csv("../data/gtfs_static/stops.txt", header=True)
trips = spark.read.csv(
    "../data/gtfs_static/trips.txt",
    header=True,
    inferSchema=True
).withColumn("route_id", col("route_id").cast("string"))
calendar = spark.read.csv(
    "../data/gtfs_static/calendar.txt",
    header=True
)
stop_times = spark.read.csv(
    "../data/gtfs_static/stop_times.txt",
    header=True
).withColumnRenamed("arrival_time", "scheduled_arrival_time") \
 .withColumnRenamed("departure_time", "scheduled_departure_time")
routes = spark.read.csv(
    "../data/gtfs_static/routes.txt",
    header=True
).withColumn("route_id", col("route_id").cast("string"))

In [29]:
# position_df.withColumn(
#  "current_datetime", 
#     F.from_unixtime(F.col("current_datetime_unix")).cast("timestamp")
# ).show(5)

In [30]:
#  (realtime_df.withColumn(
#     "current_datetime", 
#     F.from_unixtime(F.col("current_datetime_unix")).cast("timestamp")
# ).withColumn(
#     "arrival_time",
#     F.from_unixtime(F.col("arrival_time")).cast("timestamp")
# ).withColumn(
#     "departure_time",
#     F.from_unixtime(F.col("departure_time")).cast("timestamp")
# )
# .withColumn(
#     'current_datetime_vancouver',
#     F.from_utc_timestamp(F.col('current_datetime'),"America/Vancouver"))
# .withColumn(
#     'arrival_time_vancouver',
#     F.from_utc_timestamp(F.col('arrival_time'),"America/Vancouver"))
# # .withColumn(
# #     'departure_time_vancouver',
# #     F.from_utc_timestamp(F.col('departure_time'),"America/Vancouver"))

# ).show(5)


In [37]:
position_df.withColumn( "current_datetime", F.from_unixtime(F.col("current_datetime_unix")).cast("timestamp")) \
.withColumn('current_datetime_vancouver', F.timestamp_seconds(F.col('current_datetime_unix'),'America/Vancouver')).show(5)

TypeError: timestamp_seconds() takes 1 positional argument but 2 were given

In [43]:
position_df.withColumn(
    'current_datetime_vancouver',
   
        F.timestamp_seconds(F.col("current_datetime_unix")),
   
).show(5)

+--------+--------+----------+---------------------+--------+------------+----------+-------------+------------------+-------------------+---------------------+--------------+----------+-------+----------------+---------------------+------------+--------------------+------------------------+--------------------------+
|      id| trip_id|start_date|schedule_relationship|route_id|direction_id|vehicle_id|vehicle_label|          latitude|          longitude|current_stop_sequence|current_status| timestamp|stop_id|       scrape_id|current_datetime_unix|current_date|current_datetime_str|current_datetime_str_utc|current_datetime_vancouver|
+--------+--------+----------+---------------------+--------+------------+----------+-------------+------------------+-------------------+---------------------+--------------+----------+-------+----------------+---------------------+------------+--------------------+------------------------+--------------------------+
|14260442|14260442|  20250110|          

In [42]:
position_df = position_df.withColumn( "current_datetime", F.from_unixtime(F.col("current_datetime_unix")).cast("timestamp"))
weather_df = weather_df.withColumn( "nowobsTime_three_hour_bucket", F.from_unixtime(F.col("current_datetime_unix")).cast("timestamp"))

In [28]:
trips_enriched = trips.join(calendar, on="service_id", how="left") \
                      .join(routes, on="route_id", how="left")

# Enrich stop_times: first join stops, then join with trips_enriched
stop_times_enriched = stop_times.join(stops, on="stop_id", how="left") \
                                .join(trips_enriched, on="trip_id", how="left")

In [29]:
realtime_enriched = realtime_df.join(stop_times_enriched, 
                                     on=["trip_id", "stop_id", "stop_sequence", "direction_id"],
                                     how="left")

In [30]:
w_full = Window.partitionBy("trip_id", "stop_sequence", "current_date") \
               .orderBy("current_datetime") \
               .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)


realtime_enriched = realtime_enriched \
    .withColumn("last_arrival_delay", F.last("arrival_delay", ignorenulls=True).over(w_full)) \
    .withColumn("last_arrival_delay_time", F.last("current_datetime", ignorenulls=True).over(w_full)) 

In [31]:


# Get the “last record rank” by ordering descending by current_datetime:
w_desc = Window.partitionBy("trip_id", "stop_sequence", "current_date") \
               .orderBy(F.col("current_datetime").desc())

realtime_enriched = realtime_enriched.withColumn("last_record_rank", F.row_number().over(w_desc))

# Extract hour and minute from the current_datetime column
realtime_enriched = realtime_enriched \
    .withColumn("hour", F.hour("current_datetime")) \
    .withColumn("minute", F.minute("current_datetime"))

In [33]:
realtime_enriched.show(5)

[1732.042s][warning][gc,alloc] Executor task launch worker for task 2.0 in stage 44.0 (TID 68): Retried waiting for GCLocker too often allocating 16058 words


+--------+-------+-------------+------------+--------+----------+----------+---------------------+--------+----------+-------------+-------------+-------------+---------------+--------------+--------------------------+----------------+---------------------+------------+-------------------+----------------------+------------------------+-------------+-----------+-------------+-------------------+---------+---------+-------------------+---------+-----------+--------+--------------+---------+--------------------+-------------+-------+--------+-----------------+---------+-------------+---------------------+--------------------+--------+---------------+----------+--------+------+-------+---------+--------+------+--------+------+--------------------+----------+----------------+-----------+---------+---------+----------+----------------+------------------+-----------------------+----------------+----+------+----------------------------------+
| trip_id|stop_id|stop_sequence|direction_id|     

In [32]:


# =============================================================================
# Create a three‐hour “bucket” column for joining with weather data
# =============================================================================

# The 3-hour bucket is computed by flooring the unix timestamp to a multiple of 10800 (3*3600) seconds.
realtime_enriched = realtime_enriched.withColumn(
    "current_datetime_three_hour_bucket",
    F.from_unixtime(F.floor(F.unix_timestamp("current_datetime") / 10800) * 10800)
)

# If the weather DF does not already have a three-hour bucket column, add one.
if "nowobsTime_three_hour_bucket" not in weather_df.columns:
    weather_df = weather_df.withColumn(
        "nowobsTime_three_hour_bucket",
        F.from_unixtime(F.floor(F.unix_timestamp("nowobsTime") / 10800) * 10800)
    )

# =============================================================================
# Join with weather data on the three-hour bucket column
# =============================================================================

merged_df = realtime_enriched.join(
    weather_df,
    realtime_enriched.current_datetime_three_hour_bucket == weather_df.nowobsTime_three_hour_bucket,
    how="left"
)

# =============================================================================
# Simulate an as-of join to merge in position data with a 3-minute tolerance
# =============================================================================

# (1) Add a unique identifier to each realtime record so we can match later.
merged_with_id = merged_df.withColumn("realtime_id", F.monotonically_increasing_id())

# (2) For a cleaner join, rename the key columns in position_df.
#     (Adjust these renames to match your schema; here we rename a few position fields.)
position_df_renamed = position_df \
    .withColumnRenamed("current_datetime", "pos_current_datetime") \
    .withColumnRenamed("current_stop_sequence", "pos_current_stop_sequence") \
    .withColumnRenamed("latitude", "pos_vehicle_latitude") \
    .withColumnRenamed("longitude", "pos_vehicle_longitude")

# (3) In both dataframes, create unix timestamp columns.
merged_with_id = merged_with_id.withColumn("r_ts", F.unix_timestamp("current_datetime"))
position_df_renamed = position_df_renamed.withColumn("p_ts", F.unix_timestamp("pos_current_datetime"))

# (4) Join on trip_id (you can add additional keys if needed) and compute the absolute time difference.
#     We use a left join so that realtime rows with no matching position remain.
pos_matches = merged_with_id.join(position_df_renamed, on="trip_id", how="left") \
    .withColumn("time_diff", F.abs(F.col("r_ts") - F.col("p_ts")))

# (5) For each realtime record, pick the position record with the smallest time difference.
w_asof = Window.partitionBy("realtime_id").orderBy("time_diff")
pos_matches = pos_matches.withColumn("rn", F.row_number().over(w_asof))

# (6) Only keep the best match if the time difference is within 3 minutes (180 seconds).
best_pos = pos_matches.filter((F.col("rn") == 1) & (F.col("time_diff") <= 180)) \
                      .drop("rn", "time_diff", "r_ts", "p_ts")

# (7) Now left join the best position match back to the realtime (merged) data.
#     (Since best_pos comes from a join, it already includes all realtime columns; we select only the position fields we need.)
pos_fields = ["realtime_id", "pos_current_stop_sequence", "pos_vehicle_latitude", "pos_vehicle_longitude", "pos_current_datetime"]
final_df = merged_with_id.join(best_pos.select(*pos_fields), on="realtime_id", how="left")


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `nowobsTime` cannot be resolved. Did you mean one of the following? [`now.obsTime`, `now.vis`, `now.cloud`, `now.dew`, `now.icon`].;
'Project [code#1159, updateTime#1160, fxLink#1161, now.obsTime#1162, now.temp#1163, now.feelsLike#1164, now.icon#1165, now.text#1166, now.wind360#1167, now.windDir#1168, now.windScale#1169, now.windSpeed#1170, now.humidity#1171, now.precip#1172, now.pressure#1173, now.vis#1174, now.cloud#1175, now.dew#1176, refer.sources#1177, refer.license#1178, scrape_id#1179, current_datetime_unix#1180L, current_date#1181, from_unixtime(('floor((unix_timestamp('nowobsTime, yyyy-MM-dd HH:mm:ss, Some(America/Vancouver), false) / 10800)) * 10800), yyyy-MM-dd HH:mm:ss, Some(America/Vancouver)) AS nowobsTime_three_hour_bucket#2240]
+- Relation [code#1159,updateTime#1160,fxLink#1161,now.obsTime#1162,now.temp#1163,now.feelsLike#1164,now.icon#1165,now.text#1166,now.wind360#1167,now.